# Data Perparation

We will use `HFTBacktest` as both the backtesting framework and the reinforcement learning environment.
Therefore, it is essential to preprocess and convert our market data into the format required by `HFTBacktest`.
Please refer to the official documentation for data format specifications: [HFTBacktest Data Guide](https://hftbacktest.readthedocs.io/en/latest/data.html)


## Load dependencies


In [19]:
import gzip
from hftbacktest.data.utils.snapshot import create_last_snapshot
import lib.binancefutures_convertor as binancefutures


with gzip.open('data/binance/xrpusdt_20250629.gz', 'r') as f:
    for i in range(200):
        line = f.readline()
        print(line)

b'1751240969020830000 {"stream":"xrpusdt@bookTicker","data":{"e":"bookTicker","u":7910652949033,"s":"XRPUSDT","b":"2.2058","B":"8122.6","a":"2.2059","A":"31314.9","T":1751240968948,"E":1751240968948}}\n'
b'1751240969140033000 {"stream":"xrpusdt@depth@0ms","data":{"e":"depthUpdate","E":1751240968961,"T":1751240968953,"s":"XRPUSDT","U":7910652947517,"u":7910652949212,"pu":7910652947340,"b":[["2.2005","102082.0"],["2.2029","22069.2"],["2.2030","87657.8"]],"a":[["2.2059","31314.9"],["2.2110","83438.6"],["2.2111","76066.9"]]}}\n'
b'1751240969186300000 {"stream":"xrpusdt@depth@0ms","data":{"e":"depthUpdate","E":1751240969014,"T":1751240969010,"s":"XRPUSDT","U":7910652949631,"u":7910652952331,"pu":7910652949212,"b":[["1.9853","3306.6"],["2.2055","8578.2"],["2.2056","6380.6"]],"a":[["2.2110","68577.3"]]}}\n'
b'1751240969201189000 {"stream":"xrpusdt@depth@0ms","data":{"e":"depthUpdate","E":1751240969091,"T":1751240969091,"s":"XRPUSDT","U":7910652954564,"u":7910652957694,"pu":7910652952331,"b":[

## Set constantes


In [20]:
# Convert all data files (多个日期)
import os
import numpy as np

# 🔧 扩展：处理多个日期的数据
pairs = ['ethusdt', 'solusdt', 'dogeusdt', 'xrpusdt', 'xrpusdc']
# dates = ['20250629', '20250630', '20250701']  # 所有可用日期
dates = ['20250716', '20250717']  # 所有可用日期
results = {}
conversion_status = {}

print("🚀 开始批量转换币安期货数据...")
print(
    f"📊 计划处理: {len(pairs)} 个币种 × {len(dates)} 个日期 = {len(pairs) * len(dates)} 个文件")
print("-" * 70)

for date in dates:
    print(f"\n📅 处理日期: {date}")
    print("=" * 50)

    date_results = {}

    for pair in pairs:
        input_file = f'data/binance/{pair}_{date}.gz'
        output_file = f'data/output/{pair}_{date}.npz'

        # 检查输入文件是否存在
        if not os.path.exists(input_file):
            print(f"⚠️  {pair.upper()}: 输入文件不存在 ({input_file})")
            conversion_status[f"{pair}_{date}"] = "输入文件缺失"
            continue

        if os.path.exists(output_file):
            # 加载已存在的文件
            try:
                data = np.load(output_file)['data']
                event_count = len(data)
                date_results[pair] = event_count
                print(f"✅ {pair.upper()}: {event_count:,} events (已存在)")
                conversion_status[f"{pair}_{date}"] = "已存在"
            except Exception as e:
                print(f"❌ {pair.upper()}: 加载已存在文件失败 - {e}")
                conversion_status[f"{pair}_{date}"] = f"加载失败: {e}"
        else:
            # 转换新文件
            try:
                print(f"🔄 {pair.upper()}: 开始转换...")
                data = binancefutures.convert(
                    input_file,
                    output_filename=output_file,
                    combined_stream=True
                )
                event_count = len(data)
                date_results[pair] = event_count
                print(f"✅ {pair.upper()}: {event_count:,} events (新转换)")
                conversion_status[f"{pair}_{date}"] = "新转换"
            except Exception as e:
                print(f"❌ {pair.upper()}: 转换失败 - {e}")
                conversion_status[f"{pair}_{date}"] = f"转换失败: {e}"

    # 记录每日结果
    results[date] = date_results

    # 每日统计
    if date_results:
        daily_total = sum(date_results.values())
        print(f"\n📊 {date} 日统计: {daily_total:,} events ({len(date_results)} 个币种)")

# 🎯 最终汇总统计
print(f"\n" + "=" * 70)
print("🎉 批量转换完成！最终统计:")
print("=" * 70)

total_events = 0
successful_conversions = 0
total_files = 0

for date, date_results in results.items():
    if date_results:
        date_total = sum(date_results.values())
        total_events += date_total
        successful_conversions += len(date_results)
        print(f"📅 {date}: {date_total:,} events ({len(date_results)} 币种)")

        # 显示每个币种的详情
        for pair, count in date_results.items():
            print(f"   └─ {pair.upper()}: {count:,}")

print(f"\n🏆 总计统计:")
print(f"   📊 总事件数: {total_events:,}")
print(f"   ✅ 成功转换: {successful_conversions} 个文件")
print(
    f"   📁 生成文件数: {len([k for k, v in conversion_status.items() if v in ['已存在', '新转换']])} 个")

# 🔍 显示转换状态详情
print(f"\n📋 详细转换状态:")
for file_key, status in conversion_status.items():
    pair, date = file_key.split('_')
    status_icon = "✅" if status in [
        "已存在", "新转换"] else "❌" if "失败" in status or "缺失" in status else "⚠️"
    print(f"   {status_icon} {pair.upper()}_{date}: {status}")

print(f"\n💡 可用的训练数据文件:")
available_files = [k for k, v in conversion_status.items() if v in [
    '已存在', '新转换']]
for file_key in sorted(available_files):
    pair, date = file_key.split('_')
    print(f"   📁 data/output/{pair}_{date}.npz")

🚀 开始批量转换币安期货数据...
📊 计划处理: 5 个币种 × 2 个日期 = 10 个文件
----------------------------------------------------------------------

📅 处理日期: 20250716
✅ ETHUSDT: 153,265,836 events (已存在)
⚠️  SOLUSDT: 输入文件不存在 (data/binance/solusdt_20250716.gz)
⚠️  DOGEUSDT: 输入文件不存在 (data/binance/dogeusdt_20250716.gz)
⚠️  XRPUSDT: 输入文件不存在 (data/binance/xrpusdt_20250716.gz)
⚠️  XRPUSDC: 输入文件不存在 (data/binance/xrpusdc_20250716.gz)

📊 20250716 日统计: 153,265,836 events (1 个币种)

📅 处理日期: 20250717
⚠️  ETHUSDT: 输入文件不存在 (data/binance/ethusdt_20250717.gz)
⚠️  SOLUSDT: 输入文件不存在 (data/binance/solusdt_20250717.gz)
⚠️  DOGEUSDT: 输入文件不存在 (data/binance/dogeusdt_20250717.gz)
🔄 XRPUSDT: 开始转换...
Using gunzip command to decompress file...
Correcting the latency
local_timestamp is ahead of exch_timestamp by 83479000
Correcting the event order
Saving to data/output/xrpusdt_20250717.npz
✅ XRPUSDT: 82,141,803 events (新转换)
✅ XRPUSDC: 9,653,647 events (已存在)

📊 20250717 日统计: 91,795,450 events (2 个币种)

🎉 批量转换完成！最终统计:
📅 20250716: 153,265,836 events

In [21]:
# 🔧 为所有币种创建EOD快照数据
import glob
from hftbacktest.data.utils.snapshot import create_last_snapshot

# 币种交易规则配置
TRADING_RULES = {
    'ethusdt': {
        'tick_size': 0.01,
        'lot_size': 0.001,
    },
    'solusdt': {
        'tick_size': 0.0001,
        'lot_size': 0.01,
    },
    'xrpusdt': {
        'tick_size': 0.0001,
        'lot_size': 0.1,
    },
    'dogeusdt': {
        'tick_size': 0.000001,
        'lot_size': 1,
    }
}

# 为所有可用日期和币种创建EOD快照
pairs = ['ethusdt', 'solusdt', 'dogeusdt', 'xrpusdt']
# dates = [20250629, 20250630]  # 跳过20250701，因为数据不完整
dates = [20250716]  # 跳过20250701，因为数据不完整

print("🔧 开始创建EOD快照数据...")
print(
    f"📊 计划处理: {len(pairs)} 个币种 × {len(dates)} 个日期 = {len(pairs) * len(dates)} 个快照")
print("-" * 70)

eod_results = {}

for date in dates:
    print(f"\n📅 处理日期: {date}")
    print("=" * 50)

    date_results = {}

    for pair in pairs:
        input_file = f'data/output/{pair}_{date}.npz'
        eod_file = f'data/output/{pair}_{date}_eod.npz'

        # 检查输入文件是否存在
        if not os.path.exists(input_file):
            print(f"⚠️  {pair.upper()}: 输入文件不存在 ({input_file})")
            continue

        # 检查EOD文件是否已存在
        if os.path.exists(eod_file):
            print(f"✅ {pair.upper()}: EOD快照已存在")
            date_results[pair] = "已存在"
            continue

        # 创建EOD快照
        try:
            print(f"🔄 {pair.upper()}: 创建EOD快照...")
            trading_rule = TRADING_RULES[pair]

            data = create_last_snapshot(
                [input_file],
                tick_size=trading_rule['tick_size'],
                lot_size=trading_rule['lot_size'],
                output_snapshot_filename=eod_file
            )

            print(f"✅ {pair.upper()}: EOD快照创建成功")
            date_results[pair] = "新创建"

        except Exception as e:
            print(f"❌ {pair.upper()}: EOD快照创建失败 - {e}")
            date_results[pair] = f"失败: {e}"

    eod_results[date] = date_results

    # 每日统计
    if date_results:
        success_count = len(
            [v for v in date_results.values() if v in ["已存在", "新创建"]])
        print(f"\n📊 {date} 日统计: {success_count}/{len(date_results)} 个币种成功创建EOD快照")

# 🎯 最终汇总统计
print(f"\n" + "=" * 70)
print("🎉 EOD快照创建完成！最终统计:")
print("=" * 70)

total_success = 0
total_files = 0

for date, date_results in eod_results.items():
    if date_results:
        success_count = len(
            [v for v in date_results.values() if v in ["已存在", "新创建"]])
        total_success += success_count
        total_files += len(date_results)
        print(f"📅 {date}: {success_count}/{len(date_results)} 成功")

        # 显示每个币种的详情
        for pair, status in date_results.items():
            status_icon = "✅" if status in ["已存在", "新创建"] else "❌"
            print(f"   {status_icon} {pair.upper()}: {status}")

print(f"\n🏆 总计统计:")
print(f"   ✅ 成功创建: {total_success}/{total_files} 个EOD快照")

# 🔍 列出所有可用的EOD文件
print(f"\n💡 可用的EOD快照文件:")
eod_files = glob.glob('data/output/*_eod.npz')
for eod_file in sorted(eod_files):
    filename = os.path.basename(eod_file)
    pair = filename.split('_')[0].upper()
    date = filename.split('_')[1]
    print(f"   📁 {filename} ({pair} - {date})")

🔧 开始创建EOD快照数据...
📊 计划处理: 4 个币种 × 1 个日期 = 4 个快照
----------------------------------------------------------------------

📅 处理日期: 20250716
✅ ETHUSDT: EOD快照已存在
⚠️  SOLUSDT: 输入文件不存在 (data/output/solusdt_20250716.npz)
⚠️  DOGEUSDT: 输入文件不存在 (data/output/dogeusdt_20250716.npz)
⚠️  XRPUSDT: 输入文件不存在 (data/output/xrpusdt_20250716.npz)

📊 20250716 日统计: 1/1 个币种成功创建EOD快照

🎉 EOD快照创建完成！最终统计:
📅 20250716: 1/1 成功
   ✅ ETHUSDT: 已存在

🏆 总计统计:
   ✅ 成功创建: 1/1 个EOD快照

💡 可用的EOD快照文件:
   📁 dogeusdt_20250629_eod.npz (DOGEUSDT - 20250629)
   📁 dogeusdt_20250630_eod.npz (DOGEUSDT - 20250630)
   📁 ethusdt_20250629_eod.npz (ETHUSDT - 20250629)
   📁 ethusdt_20250630_eod.npz (ETHUSDT - 20250630)
   📁 ethusdt_20250716_eod.npz (ETHUSDT - 20250716)
   📁 solusdt_20250629_eod.npz (SOLUSDT - 20250629)
   📁 solusdt_20250630_eod.npz (SOLUSDT - 20250630)
   📁 xrpusdt_20250629_eod.npz (XRPUSDT - 20250629)
   📁 xrpusdt_20250630_eod.npz (XRPUSDT - 20250630)
